# Imports

In [25]:
import scipy
from pathlib import Path
import numpy as np
import pandas as pd
import trimesh
from dataclasses import dataclass
import numpy as np
from enum import Enum
import torch
from scipy.spatial.transform import Rotation
import k3d

# Generate Dataset Overview (Optional)

This cell generates the dataset_overview.csv file. You only need to run it once.

In [26]:
# DATASET_ROOT = Path("dropbox")
# SUBFOLDERS = [
#     "boxes", "labels", "models", "obbs", "ops", "part mesh indices", "syms"
# ]
# 
# rows = []
# 
# for category_path in DATASET_ROOT.iterdir():
#     if not category_path.is_dir():
#         continue
#     category_name = category_path.name
#     syms_folder = category_path / "syms"
#     if not syms_folder.exists():
#         continue
# 
#     for mat_file in syms_folder.glob("*.mat"):
#         data = scipy.io.loadmat(mat_file)
#         if "shapename" not in data:
#             continue
#         shape_name = str(data["shapename"].squeeze())
#         row_name = f"{category_name}_{mat_file.stem}"
#         row = {"shape": row_name}
# 
#         for sub in SUBFOLDERS:
#             sub_path = category_path / sub
#             if sub_path.exists():
#                 if sub == "models":
#                     obj_file = sub_path / f"{shape_name}.obj"
#                     row[sub] = str(obj_file) if obj_file.exists() else None
#                 elif sub == "obbs":
#                     obb_file = sub_path / f"{shape_name}.obb"
#                     row[sub] = str(obb_file) if obb_file.exists() else None
#                 else:
#                     candidate = sub_path / mat_file.name
#                     row[sub] = str(candidate) if candidate.exists() else None
#             else:
#                 row[sub] = None
# 
#         rows.append(row)
# 
# df = pd.DataFrame(rows).set_index("shape").sort_index()
# df.columns = [c.replace(" ", "_") for c in df.columns]
# df.to_csv("dataset_overview.csv")
# print("Saved to 'dataset_overview.csv'")

# ShapeData Class

A dataclass to load and hold data for a single shape from the dataset.

In [27]:
@dataclass
class ShapeData:
    def __init__(self, row):
        self.boxes = scipy.io.loadmat(row.boxes)["box"]
        self.labels = scipy.io.loadmat(row.labels)["label"]
        self.ops = scipy.io.loadmat(row.ops)["op"]
        self.part_mesh_indices = scipy.io.loadmat(row.part_mesh_indices)["cell_boxs_correspond_objSerialNumber"]

        sym_mat = scipy.io.loadmat(row.syms)
        self.syms = sym_mat["sym"]
        self.shapename = str(sym_mat["shapename"].squeeze())
        
        self.models = row.models
        self.obbs = row.obbs

# Load Data and Select a Shape

In [28]:
df = pd.read_csv("dataset_overview.csv", index_col="shape")
# chair_row = df.sample(n=1).iloc[0]
# chair_row = df.loc["Chair_3500"]
# chair_row = df.loc["Chair_4986"]
chair_row = df.loc["Chair_997"]
print("Selected chair:", chair_row.name)

Selected chair: Chair_997


# Display Selected Chair Data Paths

In [29]:
chair_row

boxes                            dropbox/Chair/boxes/997.mat
labels                          dropbox/Chair/labels/997.mat
models                        dropbox/Chair/models/35475.obj
obbs                            dropbox/Chair/obbs/35475.obb
ops                                dropbox/Chair/ops/997.mat
part_mesh_indices    dropbox/Chair/part mesh indices/997.mat
syms                              dropbox/Chair/syms/997.mat
Name: Chair_997, dtype: object

# Inspect Loaded Data

In [30]:
def display_section(description, data):
    print(f"\n=== {description} ===")
    print(data)

file_descriptions = {
    "boxes": "Part bounding boxes",
    "labels": "Semantic labels of parts",
    "models": "3D mesh (.obj)",
    "obbs": "Whole shape bounding box",
    "ops": "Node types (leaf, adjacency, symmetry)",
    "part mesh indices": "Indices of part meshes",
    "syms": "Symmetry parameters"
}

chair = ShapeData(chair_row)

display_section("Chair Shape Name", chair.shapename)
display_section(file_descriptions["boxes"], chair.boxes)
display_section(file_descriptions["labels"], chair.labels)
display_section(file_descriptions["models"], chair.models)
display_section(file_descriptions["obbs"], chair.obbs)
display_section(file_descriptions["ops"], chair.ops)
display_section(file_descriptions["part mesh indices"], chair.part_mesh_indices)
display_section(file_descriptions["syms"], chair.syms)


=== Chair Shape Name ===
35475

=== Part bounding boxes ===
[[-2.51200e-03  4.15613e-01 -4.32565e-01 -4.26290e-01 -4.26395e-01
  -4.26289e-01 -4.33016e-01 -2.51200e-03 -2.51200e-03 -4.19624e-01]
 [-1.02519e-01  2.01313e-01  2.01753e-01  7.85405e-02 -4.12334e-01
  -2.77617e-01 -4.28291e-01  3.86537e-01  5.22794e-01  2.58365e-01]
 [ 9.82420e-02  1.60217e-01  1.60065e-01  4.62261e-01  4.67697e-01
   9.72990e-02 -3.02368e-01 -3.15111e-01 -3.67270e-01 -3.03164e-01]
 [ 2.08916e-01  1.08473e-01  1.00663e-01  2.65593e-01  7.37226e-01
   5.16800e-02  7.12119e-01  7.97568e-02  1.65378e-01  7.32303e-01]
 [ 7.91444e-01  7.90014e-01  7.89418e-01  8.81490e-02  7.72790e-02
   6.86218e-01  9.84591e-02  9.35933e-02  8.07342e-02  1.23785e-01]
 [ 8.88898e-01  9.79786e-02  8.75671e-02  6.60900e-02  6.58790e-02
   4.13440e-02  9.77008e-02  7.85538e-01  7.85538e-01  7.77318e-02]
 [ 0.00000e+00  3.10451e-01  1.92281e-01  0.00000e+00  0.00000e+00
   0.00000e+00  2.42457e-02 -1.16452e-17 -0.00000e+00  2.59063

# Define the Tree Structure

This cell defines the Tree class to reconstruct the shape's hierarchical structure.

In [31]:
class Tree(object):
    class NodeType(Enum):
        BOX = 0
        ADJ = 1
        SYM = 2

    class Node(object):
        def __init__(self, box=None, left=None, right=None, node_type=None, sym=None, label=None):
            self.box = box
            self.sym = sym
            self.left = left
            self.right = right
            self.node_type = node_type
            self.label = label

        def is_leaf(self):
            return self.node_type == Tree.NodeType.BOX and self.box is not None

        def is_adj(self):
            return self.node_type == Tree.NodeType.ADJ

        def is_sym(self):
            return self.node_type == Tree.NodeType.SYM

    def __init__(self, boxes, ops, syms, labels):
        box_list = [b for b in torch.split(boxes, 1, 0)]
        sym_param = [s for s in torch.split(syms, 1, 0)]
        label_list = [l for l in labels[0]]
        box_list.reverse()
        sym_param.reverse()
        label_list.reverse()
        queue = []
        for id in range(ops.size()[1]):
            if ops[0, id] == Tree.NodeType.BOX.value:
                queue.append(Tree.Node(box=box_list.pop(), node_type=Tree.NodeType.BOX, label=label_list.pop()))
            elif ops[0, id] == Tree.NodeType.ADJ.value:
                left_node = queue.pop()
                right_node = queue.pop()
                queue.append(Tree.Node(left=left_node, right=right_node, node_type=Tree.NodeType.ADJ))
            elif ops[0, id] == Tree.NodeType.SYM.value:
                node = queue.pop()
                queue.append(Tree.Node(left=node, sym=sym_param.pop(), node_type=Tree.NodeType.SYM))
        self.root = queue[0]

boxes = torch.tensor(chair.boxes, dtype=torch.float).t()
ops = torch.tensor(chair.ops, dtype=torch.int)
syms = torch.tensor(chair.syms, dtype=torch.float).t()
labels = torch.tensor(chair.labels, dtype=torch.int)

tree = Tree(boxes, ops, syms, labels)

# Tree Traversal and Printing Utilities

In [32]:
def normalize_vector(vec):
    return vec / np.linalg.norm(vec)

def calculate_orthonormal_basis(dir1, dir2):
    dir1_norm = normalize_vector(dir1)
    dir2_norm = normalize_vector(dir2)
    dir3_norm = normalize_vector(np.cross(dir1_norm, dir2_norm))
    return dir1_norm, dir2_norm, dir3_norm

label_names = {
    0: "Backrest",
    1: "Seat",
    2: "Leg",
    3: "Armrest"
}

def format_box_as_dict(bv):
    center = np.round(bv[0:3], 7)
    len_y, len_z, len_x = bv[3], bv[4], bv[5]
    lengths = np.round([len_x, len_y, len_z], 7)
    raw_dir2, raw_dir3 = bv[6:9], bv[9:12]
    dir2, dir3, dir1 = calculate_orthonormal_basis(raw_dir2, raw_dir3)
    return {'center': center, 'dir1': dir1, 'dir2': dir2, 'dir3': dir3, 'lengths': lengths}

def print_box_dict(box, prefix):
    def fmt(arr):
        return "[" + ", ".join(f"{x:.7f}" for x in arr) + "]"
    print(f"{prefix}center : {fmt(box['center'])}")
    print(f"{prefix}dir1   : {fmt(box['dir1'])}")
    print(f"{prefix}dir2   : {fmt(box['dir2'])}")
    print(f"{prefix}dir3   : {fmt(box['dir3'])}")
    print(f"{prefix}lengths: {fmt(box['lengths'])}")

def print_full_tree(node, prefix="", is_last=True):
    branch = "└─ " if is_last else "├─ "
    if node.is_leaf():
        part_name = label_names.get(node.label.item(), f"Unknown({node.label.item()})")
        box_values = node.box.squeeze().tolist()
        box_dict = format_box_as_dict(box_values)
        print(f"{prefix}{branch}Leaf: {part_name}, Label={node.label.item()}")
        print_box_dict(box_dict, prefix + "    ")
    elif node.is_adj():
        print(f"{prefix}{branch}Adjacency Node")
        children = [node.left, node.right]
        for i, child in enumerate(children):
            print_full_tree(child, prefix + ("    " if is_last else "│   "), i == len(children)-1)
    elif node.is_sym():
        sym_values = node.sym.squeeze().tolist()
        print(f"{prefix}{branch}Symmetry Node: Sym Params={[round(x,6) for x in sym_values[:]]}")
        print_full_tree(node.left, prefix + ("    " if is_last else "│   "), True)

print("Chair Tree:")
print_full_tree(tree.root)

Chair Tree:
└─ Adjacency Node
    ├─ Adjacency Node
    │   ├─ Adjacency Node
    │   │   ├─ Symmetry Node: Sym Params=[0.0, 0.999976, -0.006277, 0.002913, -0.002533, 0.255747, -0.301949, 0.0]
    │   │   │   └─ Leaf: Backrest, Label=0
    │   │   │       center : [-0.4196240, 0.2583650, -0.3031640]
    │   │   │       dir1   : [0.9959217, -0.0414941, -0.0801138]
    │   │   │       dir2   : [0.0259063, 0.9820917, -0.1866139]
    │   │   │       dir3   : [0.0864224, 0.1837771, 0.9791615]
    │   │   │       lengths: [0.0777318, 0.7323030, 0.1237850]
    │   │   └─ Leaf: Backrest, Label=0
    │   │       center : [-0.0025120, 0.5227940, -0.3672700]
    │   │       dir1   : [1.0000000, 0.0000000, -0.0000000]
    │   │       dir2   : [-0.0000000, 0.9159900, -0.4012010]
    │   │       dir3   : [0.0000000, 0.4012010, 0.9159900]
    │   │       lengths: [0.7855380, 0.1653780, 0.0807342]
    │   └─ Symmetry Node: Sym Params=[1.0, -0.001884, -0.08541, 0.181399, -0.002512, 0.011224, -0.260177,

# Extract and Reconstruct Boxes from Tree

This function traverses the tree, applies symmetry operations, and returns a flat list of all component boxes.

In [33]:
def extract_boxes_and_labels_with_sym_corrected(node):
    data = {'boxes': [], 'connections': [], 'symmetries': [], 'labels': []}

    def traverse(n):
        if n.is_leaf():
            box_values = n.box.squeeze().tolist()
            box_dict = format_box_as_dict(box_values)
            box_dict['label_id'] = n.label.item()
            box_dict['label_name'] = label_names.get(n.label.item(), f"Unknown({n.label.item()})")
            data['boxes'].append(box_dict)
            data['labels'].append(n.label.item())
        elif n.is_adj():
            if hasattr(n, 'left') and n.left is not None: traverse(n.left)
            if hasattr(n, 'right') and n.right is not None: traverse(n.right)
        elif n.is_sym():
            sym_values = n.sym.squeeze().tolist()
            sym_type = sym_values[0]

            original_boxes = []
            def collect_boxes(sub_node):
                if sub_node.is_leaf():
                    box_values = sub_node.box.squeeze().tolist()
                    box_dict = format_box_as_dict(box_values)
                    box_dict['label_id'] = sub_node.label.item()
                    box_dict['label_name'] = label_names.get(sub_node.label.item(), f"Unknown({sub_node.label.item()})")
                    original_boxes.append(box_dict)
                elif sub_node.is_adj() or sub_node.is_sym():
                    if hasattr(sub_node, 'left') and sub_node.left is not None: collect_boxes(sub_node.left)
                    if hasattr(sub_node, 'right') and sub_node.right is not None: collect_boxes(sub_node.right)
            
            collect_boxes(n)
            
            for box in original_boxes:
                data['boxes'].append(box)
                data['labels'].append(box['label_id'])

            if sym_type == 0.0:
                plane_normal = np.array(sym_values[1:4])
                point_on_plane = np.array(sym_values[4:7])
                for box in original_boxes:
                    reflected_box = box.copy()
                    original_center = box['center']
                    vector_to_plane = original_center - point_on_plane
                    dot_product = np.dot(vector_to_plane, plane_normal)
                    reflected_center = original_center - 2 * dot_product * plane_normal
                    reflected_box['center'] = np.round(reflected_center, 7)
                    data['boxes'].append(reflected_box)
                    data['labels'].append(reflected_box['label_id'])
            elif sym_type == -1.0:
                rotation_axis = np.array(sym_values[1:4])
                rotation_center = np.array(sym_values[4:7])
                n_fold = int(round(1.0 / sym_values[7]))
                rotation_axis_norm = rotation_axis / np.linalg.norm(rotation_axis)
                for box in original_boxes:
                    for i in range(1, n_fold):
                        angle = 2 * np.pi * i / n_fold
                        rotated_box = box.copy()
                        original_center = box['center']
                        vector_from_center = original_center - rotation_center
                        rotation = Rotation.from_rotvec(angle * rotation_axis_norm)
                        rotated_vector = rotation.apply(vector_from_center)
                        rotated_center = rotation_center + rotated_vector
                        
                        rotated_box['center'] = np.round(rotated_center, 7)
                        rotated_box['dir1'] = np.round(rotation.apply(box['dir1']), 7)
                        rotated_box['dir2'] = np.round(rotation.apply(box['dir2']), 7)
                        rotated_box['dir3'] = np.round(rotation.apply(box['dir3']), 7)

                        data['boxes'].append(rotated_box)
                        data['labels'].append(rotated_box['label_id'])
            elif sym_type == 1.0:  # Translational symmetry
                end_point = np.array(sym_values[4:7])
                n_copies = int(round(1.0 / sym_values[7]))
                
                for box in original_boxes:
                    original_center = np.array(box['center'])
                    if n_copies > 1:
                        # Translation vector along the symmetry direction
                        direction = end_point - original_center
                        step = direction / (n_copies - 1)
                    else:
                        step = np.zeros(3)

                    # Generate all copies along the translation
                    for i in range(n_copies):
                        new_box = box.copy()
                        new_box['center'] = np.round(original_center + step * i, 7)
                        data['boxes'].append(new_box)
                        data['labels'].append(new_box['label_id'])

    traverse(node)
    return data

tree_data = extract_boxes_and_labels_with_sym_corrected(tree.root)
print(f"Total boxes: {len(tree_data['boxes'])}")

Total boxes: 20


# View Extracted Tree Data

In [34]:
tree_data

{'boxes': [{'center': array([-0.419624,  0.258365, -0.303164]),
   'dir1': array([ 0.9959217 , -0.04149407, -0.08011377]),
   'dir2': array([ 0.02590629,  0.9820917 , -0.18661394]),
   'dir3': array([0.08642244, 0.18377708, 0.97916145]),
   'lengths': array([0.0777318, 0.732303 , 0.123785 ]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': array([ 0.414557 ,  0.2531285, -0.3007339]),
   'dir1': array([ 0.9959217 , -0.04149407, -0.08011377]),
   'dir2': array([ 0.02590629,  0.9820917 , -0.18661394]),
   'dir3': array([0.08642244, 0.18377708, 0.97916145]),
   'lengths': array([0.0777318, 0.732303 , 0.123785 ]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': array([-0.002512,  0.522794, -0.36727 ]),
   'dir1': array([ 1.,  0., -0.]),
   'dir2': array([-0.        ,  0.91599003, -0.40120103]),
   'dir3': array([0.        , 0.40120103, 0.91599003]),
   'lengths': array([0.785538 , 0.165378 , 0.0807342]),
   'label_id': 0,
   'label_name': 'Backrest'},
  {'center': 

# Visualize Reconstructed Tree Data

In [35]:
LABEL_COLORS = {
    0: 0xFF0000,
    1: 0x00FF00,
    2: 0x0000FF,
    3: 0xFFFF00,
}

def plot_tree_data(tree_data):
    plot = k3d.plot()
    for box in tree_data['boxes']:
        center = box["center"]
        dir1, dir2, dir3 = box["dir1"], box["dir2"], box["dir3"]
        lengths = box["lengths"]
        label_id = box.get("label_id", -1)
        
        d1, d2, d3 = 0.5 * lengths[0] * dir1, 0.5 * lengths[1] * dir2, 0.5 * lengths[2] * dir3
        
        corners = np.array([
            center - d1 - d2 - d3, center - d1 + d2 - d3,
            center + d1 - d2 - d3, center + d1 + d2 - d3,
            center - d1 - d2 + d3, center - d1 + d2 + d3,
            center + d1 - d2 + d3, center + d1 + d2 + d3
        ], dtype=np.float32)

        faces = np.array([
            [0,1,3], [0,3,2], [4,6,7], [4,7,5],
            [0,2,6], [0,6,4], [1,5,7], [1,7,3],
            [0,4,5], [0,5,1], [2,3,7], [2,7,6]
        ], dtype=np.uint32)

        color = LABEL_COLORS.get(label_id, 0x808080)
        plot += k3d.mesh(corners, faces, color=color)
    plot.display()

plot_tree_data(tree_data)

Output()

# Inspect Raw OBB File Content

In [36]:
def print_obb_file(file_path):
    try:
        with open(file_path, 'r') as f:
            print(f.read())
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print_obb_file(chair.obbs)

N 19
-0.002512 0.522794 -0.36727 1 0 0 0 0.401201 0.91599 0 -0.91599 0.401201 0.785538 0.0807342 0.165378
-0.002512 -0.102519 0.098242 1 0 0 0 1 0 0 0 1 0.888898 0.208916 0.791444
-0.419624 0.258365 -0.303164 -0.0864224 -0.183777 -0.979161 0.995922 -0.0414941 -0.0801138 -0.0259063 -0.982092 0.186614 0.123785 0.0777318 0.732303
0.414557 0.253128 -0.300734 0.274244 -0.147553 -0.950273 0.961261 0.0705266 0.266464 0.0277019 -0.986536 0.161179 0.124937 0.0932775 0.743616
-0.002512 0.386537 -0.315111 1 1.23592e-17 1.16452e-17 -1.23592e-17 0.0594402 0.998232 1.16452e-17 -0.998232 0.0594402 0.785538 0.0935933 0.0797568
-0.002512 0.294735 -0.290998 1 0 0 0 0.172533 0.985004 0 -0.985004 0.172533 0.785538 0.0842166 0.0662743
-0.002512 0.201358 -0.273972 1 0 0 0 1 0 0 0 1 0.785538 0.073468 0.089392
-0.002512 0.106876 -0.264182 1 0 0 -0 0.0758333 0.997121 0 -0.997121 0.0758333 0.785538 0.0810064 0.0620166
-0.002512 0.011224 -0.260177 1 0 0 0 1 0 0 0 1 0.785538 0.06268 0.081512
-0.432565 0.201753 0.

# Parse the OBB File

In [37]:
def parse_obb_box_line(vals):
    return {
        'center': vals[0:3], 'dir1': vals[3:6], 'dir2': vals[6:9], 
        'dir3': vals[9:12], 'lengths': vals[12:15]
    }

def parse_obb_file(file_path):
    data = {'boxes': [], 'connections': [], 'symmetries': [], 'labels': []}
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) < 2:
            i += 1
            continue
        
        section_type, count = parts[0], int(parts[1])
        
        if section_type == 'N':
            for _ in range(count):
                i += 1
                vals = np.array(list(map(float, lines[i].split())))
                data['boxes'].append(parse_obb_box_line(vals))
        elif section_type == 'C':
            for _ in range(count):
                i += 1
                data['connections'].append(list(map(int, lines[i].split())))
        elif section_type == 'S':
            for _ in range(count):
                sym_block = []
                i += 1
                while i < len(lines) and lines[i].split()[0] not in ['N','C','S','L']:
                    sym_block.append(lines[i])
                    i += 1
                i -= 1
                data['symmetries'].append(sym_block)
        elif section_type == 'L':
            for _ in range(count):
                i += 1
                data['labels'].append(int(lines[i]))
        i += 1
        
    for b, lbl in zip(data['boxes'], data['labels']):
        b['label_id'] = lbl
        
    return data

obb_data = parse_obb_file(chair.obbs)
obb_data

{'boxes': [{'center': array([-0.002512,  0.522794, -0.36727 ]),
   'dir1': array([1., 0., 0.]),
   'dir2': array([0.      , 0.401201, 0.91599 ]),
   'dir3': array([ 0.      , -0.91599 ,  0.401201]),
   'lengths': array([0.785538 , 0.0807342, 0.165378 ]),
   'label_id': 0},
  {'center': array([-0.002512, -0.102519,  0.098242]),
   'dir1': array([1., 0., 0.]),
   'dir2': array([0., 1., 0.]),
   'dir3': array([0., 0., 1.]),
   'lengths': array([0.888898, 0.208916, 0.791444]),
   'label_id': 1},
  {'center': array([-0.419624,  0.258365, -0.303164]),
   'dir1': array([-0.0864224, -0.183777 , -0.979161 ]),
   'dir2': array([ 0.995922 , -0.0414941, -0.0801138]),
   'dir3': array([-0.0259063, -0.982092 ,  0.186614 ]),
   'lengths': array([0.123785 , 0.0777318, 0.732303 ]),
   'label_id': 0},
  {'center': array([ 0.414557,  0.253128, -0.300734]),
   'dir1': array([ 0.274244, -0.147553, -0.950273]),
   'dir2': array([0.961261 , 0.0705266, 0.266464 ]),
   'dir3': array([ 0.0277019, -0.986536 ,  0

# Visualize Parsed OBB Data

In [38]:
def plot_obb_data(obb_data):
    plot = k3d.plot()
    for box in obb_data['boxes']:
        center = box["center"]
        dir1, dir2, dir3 = box["dir1"], box["dir2"], box["dir3"]
        lengths = box["lengths"]
        label_id = box.get("label_id", -1)
        
        d1, d2, d3 = 0.5 * lengths[0] * dir1, 0.5 * lengths[1] * dir2, 0.5 * lengths[2] * dir3
        
        corners = np.array([
            center - d1 - d2 - d3, center - d1 + d2 - d3,
            center + d1 - d2 - d3, center + d1 + d2 - d3,
            center - d1 - d2 + d3, center - d1 + d2 + d3,
            center + d1 - d2 + d3, center + d1 + d2 + d3
        ], dtype=np.float32)

        faces = np.array([
            [0,1,3], [0,3,2], [4,6,7], [4,7,5],
            [0,2,6], [0,6,4], [1,5,7], [1,7,3],
            [0,4,5], [0,5,1], [2,3,7], [2,7,6]
        ], dtype=np.uint32)

        color = LABEL_COLORS.get(label_id, 0x808080)
        plot += k3d.mesh(corners, faces, color=color)
    plot.display()

plot_obb_data(obb_data)

Output()

# Visualize 3D Mesh with Trimesh

In [39]:
mesh = trimesh.load(chair.models, force='mesh')
mesh.show()

In [40]:
# Quat Tree

In [41]:
from scipy.spatial.transform import Rotation
import numpy as np

def normalize_vector(vec):
    """Ensures a vector has a length of 1."""
    norm = np.linalg.norm(vec)
    if norm == 0:
       return vec
    return vec / norm

def create_orthonormal_basis(primary_vec, secondary_vec):
    """Creates a robust orthonormal basis from two vectors."""
    dir2 = normalize_vector(primary_vec)
    dir1 = normalize_vector(np.cross(dir2, secondary_vec))
    dir3 = np.cross(dir1, dir2)
    return dir1, dir2, dir3

label_names = {
    0: "Backrest",
    1: "Seat",
    2: "Leg",
    3: "Armrest"
}

def format_box_as_dict(bv):
    """Formats raw box values into a dictionary, including a quaternion."""
    center = np.round(bv[0:3], 7)
    len_y, len_z, len_x = bv[3], bv[4], bv[5]
    lengths = np.round([len_x, len_y, len_z], 7)
    
    raw_dir2, raw_dir3 = bv[6:9], bv[9:12]
    dir1, dir2, dir3 = create_orthonormal_basis(raw_dir2, raw_dir3)
    
    rotation_matrix = np.array([dir1, dir2, dir3]).T
    rot = Rotation.from_matrix(rotation_matrix)
    quat_xyzw = rot.as_quat()

    return {
        'center': center,
        'lengths': lengths,
        'quaternion': quat_xyzw
        # dir vectors are no longer needed in the final dict
    }

def print_box_dict(box, prefix):
    """Prints the formatted details of a bounding box, hiding direction vectors."""
    def fmt(arr):
        return "[" + ", ".join(f"{x:.7f}" for x in arr) + "]"
    
    print(f"{prefix}center    : {fmt(box['center'])}")
    # print(f"{prefix}dir1      : {fmt(box['dir1'])}") # Removed
    # print(f"{prefix}dir2      : {fmt(box['dir2'])}") # Removed
    # print(f"{prefix}dir3      : {fmt(box['dir3'])}") # Removed
    print(f"{prefix}lengths   : {fmt(box['lengths'])}")
    print(f"{prefix}quat [xyzw]: {fmt(box['quaternion'])}")

def print_full_tree(node, prefix="", is_last=True):
    """Recursively prints the entire hierarchical tree."""
    branch = "└─ " if is_last else "├─ "
    if node.is_leaf():
        part_name = label_names.get(node.label.item(), f"Unknown({node.label.item()})")
        box_values = node.box.squeeze().tolist()
        box_dict = format_box_as_dict(box_values)
        print(f"{prefix}{branch}Leaf: {part_name}, Label={node.label.item()}")
        print_box_dict(box_dict, prefix + "    ")
    elif node.is_adj():
        print(f"{prefix}{branch}Adjacency Node")
        children = [node.left, node.right]
        for i, child in enumerate(children):
            print_full_tree(child, prefix + ("    " if is_last else "│   "), i == len(children)-1)
    elif node.is_sym():
        sym_values = node.sym.squeeze().tolist()
        print(f"{prefix}{branch}Symmetry Node: Sym Params={[round(x,6) for x in sym_values[:]]}")
        print_full_tree(node.left, prefix + ("    " if is_last else "│   "), True)

# Assuming 'tree' is already defined and loaded
print("Chair Tree:")
print_full_tree(tree.root)

Chair Tree:
└─ Adjacency Node
    ├─ Adjacency Node
    │   ├─ Adjacency Node
    │   │   ├─ Symmetry Node: Sym Params=[0.0, 0.999976, -0.006277, 0.002913, -0.002533, 0.255747, -0.301949, 0.0]
    │   │   │   └─ Leaf: Backrest, Label=0
    │   │   │       center    : [-0.4196240, 0.2583650, -0.3031640]
    │   │   │       lengths   : [0.0777318, 0.7323030, 0.1237850]
    │   │   │       quat [xyzw]: [-0.0930975, 0.0418587, -0.0169410, 0.9946324]
    │   │   └─ Leaf: Backrest, Label=0
    │   │       center    : [-0.0025120, 0.5227940, -0.3672700]
    │   │       lengths   : [0.7855380, 0.1653780, 0.0807342]
    │   │       quat [xyzw]: [-0.2049512, 0.0000000, 0.0000000, 0.9787722]
    │   └─ Symmetry Node: Sym Params=[1.0, -0.001884, -0.08541, 0.181399, -0.002512, 0.011224, -0.260177, 0.2]
    │       └─ Leaf: Backrest, Label=0
    │           center    : [-0.0025120, 0.3865370, -0.3151110]
    │           lengths   : [0.7855380, 0.0797568, 0.0935933]
    │           quat [xyzw]: [-0.0

In [42]:
def expand_tree_with_symmetries(node):
    """
    Recursively traverses the tree, applies symmetry operations,
    and returns a flat list of all final box components.
    """

    # Base case: If the node is a leaf, format it and return as a single-item list.
    if node.is_leaf():
        box_values = node.box.squeeze().tolist()
        box_dict = format_box_as_dict(box_values) # Uses our updated formatter
        box_dict['label_id'] = node.label.item()
        box_dict['label_name'] = label_names.get(node.label.item(), f"Unknown({node.label.item()})")
        return [box_dict]

    # Recursive step for adjacency nodes.
    elif node.is_adj():
        left_boxes = expand_tree_with_symmetries(node.left) if node.left else []
        right_boxes = expand_tree_with_symmetries(node.right) if node.right else []
        return left_boxes + right_boxes

    # Recursive step for symmetry nodes.
    elif node.is_sym():
        # First, get the list of original component boxes from the child sub-tree.
        child_boxes = expand_tree_with_symmetries(node.left) if node.left else []
        
        sym_values = node.sym.squeeze().tolist()
        sym_type = sym_values[0]
        generated_boxes = []

        # Case 1: Reflection Symmetry
        if sym_type == 0.0:
            plane_normal = np.array(sym_values[1:4])
            point_on_plane = np.array(sym_values[4:7])
            
            for box in child_boxes:
                reflected_box = box.copy()
                
                # Reflect the center point
                vec_to_plane = box['center'] - point_on_plane
                reflected_center = box['center'] - 2 * np.dot(vec_to_plane, plane_normal) * plane_normal
                reflected_box['center'] = np.round(reflected_center, 7)

                # Reflect the orientation (original implementation missed this)
                original_rot = Rotation.from_quat(box['quaternion'])
                R_orig = original_rot.as_matrix()
                
                # Reflection matrix for orientation
                M = np.identity(3) - 2 * np.outer(plane_normal, plane_normal)
                R_new = M @ R_orig
                
                # The reflection creates a left-handed system. We flip one axis
                # to restore a right-handed system, which is required for quaternions.
                R_new[:, 0] *= -1 
                
                reflected_rot = Rotation.from_matrix(R_new)
                reflected_box['quaternion'] = reflected_rot.as_quat()
                
                generated_boxes.append(reflected_box)

        # Case 2: Rotational Symmetry
        elif sym_type == -1.0:
            axis = normalize_vector(np.array(sym_values[1:4]))
            center = np.array(sym_values[4:7])
            n_fold = int(round(1.0 / sym_values[7]))
            
            for i in range(1, n_fold):
                angle = 2 * np.pi * i / n_fold
                # Create the symmetry rotation as a quaternion
                symmetry_quat = Rotation.from_rotvec(angle * axis).as_quat()

                for box in child_boxes:
                    rotated_box = box.copy()
                    
                    # Rotate the center point
                    vec_from_center = box['center'] - center
                    rotated_vec = Rotation.from_quat(symmetry_quat).apply(vec_from_center)
                    rotated_center = center + rotated_vec
                    rotated_box['center'] = np.round(rotated_center, 7)
                    
                    # Combine the orientations using quaternion multiplication
                    original_quat = box['quaternion']
                    new_quat = (Rotation.from_quat(symmetry_quat) * Rotation.from_quat(original_quat)).as_quat()
                    rotated_box['quaternion'] = new_quat

                    generated_boxes.append(rotated_box)

        # Return the original boxes plus all the newly generated symmetric copies
        return child_boxes + generated_boxes
    
    return [] # Return empty list for any other case

# --- How to use it ---
tree_data = expand_tree_with_symmetries(tree.root)
print(f"Total boxes after expansion: {len(tree_data)}")

Total boxes after expansion: 15


In [43]:
tree_data

[{'center': array([-0.419624,  0.258365, -0.303164]),
  'lengths': array([0.0777318, 0.732303 , 0.123785 ]),
  'quaternion': array([-0.09309755,  0.04185873, -0.01694102,  0.99463244]),
  'label_id': 0,
  'label_name': 'Backrest'},
 {'center': array([ 0.414557 ,  0.2531285, -0.3007339]),
  'lengths': array([0.0777318, 0.732303 , 0.123785 ]),
  'quaternion': array([-0.09340743, -0.04417075,  0.01042584,  0.99459303]),
  'label_id': 0,
  'label_name': 'Backrest'},
 {'center': array([-0.002512,  0.522794, -0.36727 ]),
  'lengths': array([0.785538 , 0.165378 , 0.0807342]),
  'quaternion': array([-0.20495118,  0.        ,  0.        ,  0.9787722 ]),
  'label_id': 0,
  'label_name': 'Backrest'},
 {'center': array([-0.002512,  0.386537, -0.315111]),
  'lengths': array([0.785538 , 0.0797568, 0.0935933]),
  'quaternion': array([-2.97332415e-02, -6.00374221e-18,  6.00376438e-18,  9.99557869e-01]),
  'label_id': 0,
  'label_name': 'Backrest'},
 {'center': array([-0.433016, -0.428291, -0.302368]),

In [44]:
from scipy.spatial.transform import Rotation
import k3d
import numpy as np

LABEL_COLORS = {
    0: 0xFF0000, # Red: Backrest
    1: 0x00FF00, # Green: Seat
    2: 0x0000FF, # Blue: Leg
    3: 0xFFFF00, # Yellow: Armrest
}

def plot_reconstructed_boxes(box_data_list):
    """
    Plots a list of box dictionaries using their center, lengths, and quaternion.
    """
    plot = k3d.plot(name="Reconstructed Chair")
    
    for box in box_data_list:
        center = box["center"]
        lengths = box["lengths"]
        quaternion = box["quaternion"]
        label_id = box.get("label_id", -1)

        # 1. Reconstruct orientation vectors from the quaternion
        rotation = Rotation.from_quat(quaternion)
        rotation_matrix = rotation.as_matrix()
        dir1 = rotation_matrix[:, 0]
        dir2 = rotation_matrix[:, 1]
        dir3 = rotation_matrix[:, 2]

        # 2. Calculate the 8 corners of the oriented box
        d1 = 0.5 * lengths[0] * dir1
        d2 = 0.5 * lengths[1] * dir2
        d3 = 0.5 * lengths[2] * dir3
        
        corners = np.array([
            center - d1 - d2 - d3, center - d1 + d2 - d3,
            center + d1 - d2 - d3, center + d1 + d2 - d3,
            center - d1 - d2 + d3, center - d1 + d2 + d3,
            center + d1 - d2 + d3, center + d1 + d2 + d3
        ], dtype=np.float32)

        # 3. Define the 12 triangular faces of the box
        faces = np.array([
            [0,1,3], [0,3,2], [4,6,7], [4,7,5],
            [0,2,6], [0,6,4], [1,5,7], [1,7,3],
            [0,4,5], [0,5,1], [2,3,7], [2,7,6]
        ], dtype=np.uint32)

        # Add the box mesh to the plot with its semantic color
        color = LABEL_COLORS.get(label_id, 0x808080) # Default to gray
        plot += k3d.mesh(corners, faces, color=color, opacity=0.8)

    plot.display()

# --- How to use it ---
# Assuming 'tree_data' is the dictionary from the previous step
plot_reconstructed_boxes(tree_data)

Output()

In [ ]:
# import pandas as pd
# import scipy.io
# import numpy as np

# def analyze_symmetries_from_dataset(overview_csv_path):
#     """
#     Analyzes symmetry information for all shapes in the dataset overview.

#     Args:
#         overview_csv_path (str): Path to the 'dataset_overview.csv' file.

#     Returns:
#         pd.DataFrame: A DataFrame with shape names and their symmetry info.
#     """
#     # Load the dataset overview
#     try:
#         df = pd.read_csv(overview_csv_path, index_col="shape")
#     except FileNotFoundError:
#         print(f"Error: The file '{overview_csv_path}' was not found.")
#         return None

#     results = []
    
#     # Iterate over each shape in the DataFrame
#     for shape_name, row in df.iterrows():
#         syms_path = row.get('syms')

#         # Check if a path to the syms file exists
#         if pd.isna(syms_path):
#             symmetry_info = "No Syms File"
#         else:
#             try:
#                 # Load the .mat file for symmetry
#                 sym_mat = scipy.io.loadmat(syms_path)
                
#                 # Check if the 'sym' key exists and is not empty
#                 if 'sym' in sym_mat and sym_mat['sym'].size > 0:
#                     # The sym types are the first row of the 'sym' array
#                     # We convert it to a list for clean display
#                     sym_types = sym_mat['sym'][0, :].tolist()
#                     symmetry_info = sym_types
#                 else:
#                     symmetry_info = "No Symmetry Node"
#             except FileNotFoundError:
#                 symmetry_info = "Syms File Not Found"
#             except Exception as e:
#                 symmetry_info = f"Error loading file: {e}"

#         results.append({'Shape': shape_name, 'Symmetry Info': set(symmetry_info)})

#     # Create a new DataFrame from the results
#     return pd.DataFrame(results)

# # --- Run the analysis ---
# # Make sure your 'dataset_overview.csv' is in the same directory or provide the full path
# analysis_df = analyze_symmetries_from_dataset("dataset_overview.csv")

# if analysis_df is not None:
#     print("Symmetry Analysis Results:")
#     print(analysis_df.to_string()) # .to_string() ensures all rows are printed

Symmetry Analysis Results:
           Shape     Symmetry Info
0          Bag_1               {0}
1         Bag_10               {0}
2        Bag_100               {0}
3        Bag_101               {0}
4        Bag_102             {0.0}
5        Bag_103             {0.0}
6        Bag_104               {0}
7        Bag_105               {0}
8        Bag_106             {0.0}
9        Bag_107               {0}
10       Bag_108             {0.0}
11       Bag_109             {0.0}
12        Bag_11               {0}
13       Bag_110             {0.0}
14       Bag_111             {0.0}
15       Bag_112             {0.0}
16       Bag_113               {0}
17       Bag_114               {0}
18       Bag_115               {0}
19       Bag_116               {0}
20       Bag_117             {0.0}
21       Bag_118             {0.0}
22       Bag_119               {0}
23        Bag_12               {0}
24       Bag_120               {0}
25       Bag_121             {0.0}
26       Bag_122            

In [46]:
import numpy as np
from scipy.spatial.transform import Rotation

import numpy as np
from scipy.spatial.transform import Rotation

def normalize_vector(vec):
    """Ensures a vector has a length of 1."""
    norm = np.linalg.norm(vec)
    if norm == 0:
       return vec
    return vec / norm

def format_box_with_quaternion(bv):
    """
    Takes a raw box data vector and returns a dictionary with its center, 
    lengths, and a quaternion for orientation, rounded to a specified precision.
    
    Args:
        bv (list or np.array): The 12-element raw box vector.
        
    Returns:
        dict: A dictionary containing the formatted box properties.
    """
    # 1. Extract raw data from the input vector
    center = bv[0:3]
    lengths = [bv[5], bv[3], bv[4]] # Reorder to len_x, len_y, len_z
    raw_dir2, raw_dir3 = bv[6:9], bv[9:12]

    # 2. Create a robust orthonormal basis (dir vectors)
    dir2 = normalize_vector(raw_dir2)
    dir1 = normalize_vector(np.cross(dir2, raw_dir3))
    dir3 = np.cross(dir1, dir2)

    # 3. Convert the basis vectors (rotation matrix) to a quaternion
    rotation_matrix = np.array([dir1, dir2, dir3]).T
    quaternion = Rotation.from_matrix(rotation_matrix).as_quat()

    # 4. Return the formatted dictionary, rounding to 4 decimal places
    return {
        'center': np.round(center, 4).tolist(),
        'lengths': np.round(lengths, 4).tolist(),
        'quaternion': np.round(quaternion, 4).tolist()
    }

def flatten_tree_to_arrays(node, array_list=None):
    """
    Recursively traverses the tree and flattens it into a list of arrays.

    - SYM nodes become 8-element NumPy arrays.
    - LEAF nodes become 10-element NumPy arrays (center, lengths, quaternion).
    - ADJ nodes are traversed but do not produce an array themselves.
    """
    if array_list is None:
        array_list = []

    if node.is_sym():
        # Append the 8-element symmetry parameter array
        sym_array = np.round(node.sym.squeeze().numpy(), 4)
        array_list.append(sym_array)
        # Continue traversing the child of the symmetry node
        flatten_tree_to_arrays(node.left, array_list)

    elif node.is_leaf():
        # Format the box to get center, lengths, and quaternion
        box_data = node.box.squeeze().numpy()
        formatted_box = format_box_with_quaternion(box_data) # Uses the function from our last step

        # Concatenate into a single 10-element array
        leaf_array = np.concatenate([
            formatted_box['center'],
            formatted_box['lengths'],
            formatted_box['quaternion']
        ])
        array_list.append(leaf_array)

    elif node.is_adj():
        # Adjacency nodes are structural; just traverse their children
        flatten_tree_to_arrays(node.left, array_list)
        flatten_tree_to_arrays(node.right, array_list)

    return array_list

# --- Example Usage ---
# Assuming 'tree' is your fully constructed Tree object
flattened_list = flatten_tree_to_arrays(tree.root)

# --- Print the results to verify ---
for item in flattened_list:
    print(f"Array of length {len(item)}: {item}\n")

Array of length 8: [ 0.      1.     -0.0063  0.0029 -0.0025  0.2557 -0.3019  0.    ]

Array of length 10: [-0.41960001  0.25839999 -0.30320001  0.0777      0.73229998  0.1238
 -0.0931      0.0419     -0.0169      0.9946    ]

Array of length 10: [-0.0025      0.52280003 -0.3673      0.78549999  0.1654      0.0807
 -0.205       0.          0.          0.9788    ]

Array of length 8: [ 1.     -0.0019 -0.0854  0.1814 -0.0025  0.0112 -0.2602  0.2   ]

Array of length 10: [-0.0025      0.3865     -0.31510001  0.78549999  0.0798      0.0936
 -0.0297     -0.          0.          0.9996    ]

Array of length 8: [ 0.      0.9995 -0.0304  0.0013 -0.0026 -0.4252  0.4683  0.    ]

Array of length 10: [-0.433      -0.42829999 -0.30239999  0.0977      0.71210003  0.0985
  0.0584     -0.3046     -0.0315      0.9502    ]

Array of length 10: [-0.42629999 -0.27759999  0.0973      0.0413      0.0517      0.68620002
  0.          0.          0.          1.        ]

Array of length 10: [-0.42640001 -0.41

In [47]:
label_names = {0: "Backrest", 1: "Seat", 2: "Leg", 3: "Armrest"}

def normalize_vector(vec):
    """Ensures a vector has a length of 1, handling zero-length vectors."""
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

def format_box_with_quaternion(bv):
    """
    Takes a raw box data vector and returns a dictionary with its center, 
    lengths, and a quaternion for orientation, rounded to 4 decimal places.
    """
    center = bv[0:3]
    lengths = [bv[5], bv[3], bv[4]]  # Reorder to len_x, len_y, len_z
    raw_dir2, raw_dir3 = bv[6:9], bv[9:12]

    dir2 = normalize_vector(raw_dir2)
    dir1 = normalize_vector(np.cross(dir2, raw_dir3))
    dir3 = np.cross(dir1, dir2)

    rotation_matrix = np.array([dir1, dir2, dir3]).T
    quaternion = Rotation.from_matrix(rotation_matrix).as_quat()

    return {
        'center': np.round(center, 4).tolist(),
        'lengths': np.round(lengths, 4).tolist(),
        'quaternion': np.round(quaternion, 4).tolist()
    }

def serialize_tree_to_dict(node):
    """
    Recursively walks the Tree and converts it into a nested Python dictionary
    that is perfectly structured for JSON output.
    """
    if node.is_leaf():
        box_data = node.box.squeeze().numpy()
        formatted_box = format_box_with_quaternion(box_data)
        return {
            "type": "leaf",
            "label_id": node.label.item(),
            "label": label_names.get(node.label.item(), "Unknown"),
            **formatted_box
        }
    
    elif node.is_adj():
        return {
            "type": "adjacency",
            "children": [
                serialize_tree_to_dict(node.left),
                serialize_tree_to_dict(node.right)
            ]
        }
        
    elif node.is_sym():
        return {
            "type": "symmetry",
            "params": np.round(node.sym.squeeze().numpy(), 4).tolist(),
            "child": serialize_tree_to_dict(node.left)
        }
    return {}

In [48]:
# Serialize the tree into a dictionary
shape_dict = serialize_tree_to_dict(tree.root)
        
# Convert the dictionary to a JSON formatted string
json_output = json.dumps(shape_dict, indent=2)

NameError: name 'json' is not defined

In [49]:
shape_dict

{'type': 'adjacency',
 'children': [{'type': 'adjacency',
   'children': [{'type': 'adjacency',
     'children': [{'type': 'symmetry',
       'params': [0.0,
        1.0,
        -0.006300000008195639,
        0.002899999963119626,
        -0.0024999999441206455,
        0.2556999921798706,
        -0.3018999993801117,
        0.0],
       'child': {'type': 'leaf',
        'label_id': 0,
        'label': 'Backrest',
        'center': [-0.4196000099182129,
         0.25839999318122864,
         -0.30320000648498535],
        'lengths': [0.07769999653100967,
         0.7322999835014343,
         0.12380000203847885],
        'quaternion': [-0.0931, 0.0419, -0.0169, 0.9946]}},
      {'type': 'leaf',
       'label_id': 0,
       'label': 'Backrest',
       'center': [-0.0024999999441206455,
        0.5228000283241272,
        -0.36730000376701355],
       'lengths': [0.7854999899864197,
        0.16539999842643738,
        0.08070000261068344],
       'quaternion': [-0.205, 0.0, 0.0, 0.978